# Logistic Regression Training

In [ ]:
# The versions are pinned to requirements.txt. scikit-learn and librosa matter
# most: a different scikit-learn can fail to unpickle the model in the app, and
# a different librosa can produce slightly different feature values.
!pip install -q scikit-learn==1.9.1 librosa==1.0.0 soundfile==0.14.0 joblib==1.6.0

print()
print("If Colab shows a 'Restart runtime' button above, click it, then carry on")
print("from the Drive mount cell. Do not run this install cell again.")

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
# The only line to change. It must point at the project folder on your Drive,
# the one containing app.py, config/, feature_extraction/ and audio_dataset/.
PROJECT_PATH = '/content/drive/MyDrive/SonicSentinel'

In [ ]:
import os
import sys

if not os.path.isdir(PROJECT_PATH):
    raise RuntimeError(
        f"PROJECT_PATH does not exist: {PROJECT_PATH}\n"
        "Check the folder name in your Drive and fix PROJECT_PATH in the cell above."
    )

if not os.path.isfile(os.path.join(PROJECT_PATH, 'feature_extraction', 'features.py')):
    raise RuntimeError(
        f"{PROJECT_PATH} does not look like the project folder.\n"
        "It must contain app.py, config/, feature_extraction/ and src/."
    )

if PROJECT_PATH not in sys.path:
    sys.path.insert(0, PROJECT_PATH)

# Import the same modules the Flask app uses. If this fails, stop and fix it.
# Defining features in the notebook instead would train the model on vectors the
# app never produces, and the model would score well here and behave randomly
# in the application.
try:
    import feature_extraction.features as features_module
    from feature_extraction.features import FEATURE_NAMES, FEATURE_VECTOR_LENGTH
    from config.config import CLASSES, PYTHON_MODELS
    from src.training import run_training
except Exception as error:
    raise RuntimeError(
        "Could not import the project modules from PROJECT_PATH.\n"
        "Check that the whole project folder was uploaded to Drive, including "
        "config/, feature_extraction/, audio_preprocessing/ and src/."
    ) from error

# Make sure the features really came from Drive and not from some other copy.
if not os.path.abspath(features_module.__file__).startswith(os.path.abspath(PROJECT_PATH)):
    raise RuntimeError(
        f"features.py was imported from {features_module.__file__}, "
        f"not from {PROJECT_PATH}. Fix sys.path before training."
    )

print("imported the project modules from", PROJECT_PATH)
print("feature module:", features_module.__file__)
print("feature vector length:", FEATURE_VECTOR_LENGTH)
print("classes:", len(CLASSES))

In [ ]:
from sklearn.linear_model import LogisticRegression

MODEL_NAME = "logistic_regression"

estimator = LogisticRegression(
    class_weight="balanced",
    max_iter=2000,
    random_state=42,
)

param_grid = {
    "C": [0.01, 0.1, 1, 10, 100],
    "solver": ["lbfgs", "saga"],
}

In [ ]:
# Prints the split sizes and the per-class counts first, so you can check the
# data loaded correctly before the long part starts.
#
# Extracted features are cached to PROJECT_PATH/data/metadata/*.csv on Drive.
# If the runtime disconnects, run this cell again and it reuses the cache
# instead of re-extracting everything.
model = run_training(MODEL_NAME, estimator, param_grid)

In [ ]:
# What this notebook wrote to Drive.
import os

for folder, names in [
    ('python_models', [
        PYTHON_MODELS[MODEL_NAME]['model_file'],
        PYTHON_MODELS[MODEL_NAME]['scaler_file'],
        'label_encoder.pkl',
    ]),
    ('reports', [PYTHON_MODELS[MODEL_NAME]['metrics_file']]),
]:
    for name in names:
        path = os.path.join(PROJECT_PATH, folder, name)
        state = 'saved' if os.path.exists(path) else 'MISSING'
        print(f"{state:8} {folder}/{name}")

print()
print("Download those files from Drive and put them in the same folders in the")
print("project, then set ACTIVE_PYTHON_MODEL in config/config.py to the model")
print("you picked.")